# 1.2 — Limpieza y Normalización de Datos

## Unidad 1: Introducción a Desarrollo de Productos de Datos

Un producto de datos es tan bueno como los datos que consume. En este notebook trabajaremos con un dataset que tiene los problemas más comunes de la vida real: valores nulos, registros duplicados, valores atípicos y escalas inconsistentes.

### Contenido:
1. Diagnóstico inicial del dataset
2. Valores nulos — detección y tratamiento
3. Duplicados — identificación y eliminación
4. Valores atípicos — detección y decisión
5. Normalización y encoding de variables

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================

import pandas as pd
import numpy as np

# Configuración de visualización de pandas
# max_columns=None muestra todas las columnas sin truncar
pd.set_option("display.max_columns", None)
# max_rows limita la cantidad de filas visibles
pd.set_option("display.max_rows", 20)

---
## 0. Crear el dataset de ejemplo

Vamos a construir un dataset con problemas intencionales para practicar cada técnica de limpieza. Simula registros de ventas de una empresa con errores típicos de captura manual y fuentes heterogéneas.

In [ ]:
# ============================================================
# DATASET CON PROBLEMAS INTENCIONALES
# ============================================================

np.random.seed(42)  # Semilla para reproducibilidad

n = 200

# Generar datos base
datos = {
    "id_venta": list(range(1, n + 1)),
    "fecha": pd.date_range("2025-01-01", periods=n, freq="D"),
    "producto": np.random.choice(
        ["Dashboard", "Reporte", "API", "App Web", "dashboard", "REPORTE", None],
        size=n
    ),
    "region": np.random.choice(
        ["Eje Cafetero", "Bogota", "Bogotá", "Medellín", "Medellin", "Cali", None],
        size=n
    ),
    "vendedor": np.random.choice(
        ["Ana García", "Carlos López", "María Ruiz", "Pedro Martín", None],
        size=n
    ),
    "unidades": np.random.randint(1, 50, size=n).astype(float),
    "precio_unitario": np.round(np.random.uniform(50, 500, size=n), 2),
    "calificacion": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n),
}

df = pd.DataFrame(datos)

# --- Inyectar problemas adicionales ---

# Nulos dispersos en unidades y precio
indices_nulos_unidades = np.random.choice(n, size=15, replace=False)
df.loc[indices_nulos_unidades, "unidades"] = np.nan

indices_nulos_precio = np.random.choice(n, size=10, replace=False)
df.loc[indices_nulos_precio, "precio_unitario"] = np.nan

# Valores atípicos en precio_unitario (errores de captura)
df.loc[5, "precio_unitario"] = 9500.0    # Alguien agregó un cero de más
df.loc[42, "precio_unitario"] = -200.0   # Valor negativo imposible
df.loc[99, "precio_unitario"] = 12000.0  # Fuera de rango
df.loc[150, "unidades"] = 500.0          # Cantidad absurda

# Duplicados exactos (copiar filas existentes)
filas_duplicadas = df.iloc[[10, 10, 25, 25, 80]].copy()
df = pd.concat([df, filas_duplicadas], ignore_index=True)

# Columna de ingreso total (calculada, incluye efecto de nulos)
df["ingreso_total"] = df["unidades"] * df["precio_unitario"]

print(f"Dataset creado: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head(10)

---
## 1. Diagnóstico inicial

Antes de tocar los datos, necesitamos entender qué tenemos. El diagnóstico responde tres preguntas: qué forma tienen los datos, qué tipos tiene cada columna y dónde están los problemas.

In [ ]:
# ============================================================
# FORMA Y TIPOS
# ============================================================

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print()

# .info() muestra tipo de dato y conteo de valores no nulos por columna
# Si el conteo de non-null es menor que el total de filas, hay nulos
df.info()

In [ ]:
# ============================================================
# RESUMEN DE NULOS
# ============================================================

# .isnull().sum() cuenta los nulos por columna
nulos = df.isnull().sum()

# Calcular el porcentaje de nulos respecto al total de filas
porcentaje_nulos = (nulos / len(df) * 100).round(2)

# Crear un resumen combinando conteo y porcentaje
resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": porcentaje_nulos
})

# Mostrar solo columnas que tienen al menos un nulo
resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False)

In [ ]:
# ============================================================
# RESUMEN DE DUPLICADOS
# ============================================================

# .duplicated() retorna True para cada fila que es duplicada de otra anterior
n_duplicados = df.duplicated().sum()
print(f"Filas duplicadas exactas: {n_duplicados}")

# Ver cuáles son
if n_duplicados > 0:
    print("\nFilas duplicadas:")
    # keep=False marca TODAS las ocurrencias (original + copias)
    duplicados = df[df.duplicated(keep=False)].sort_values("id_venta")
    print(duplicados)

In [ ]:
# ============================================================
# VALORES UNICOS EN COLUMNAS CATEGORICAS
# ============================================================

# Verificar inconsistencias en texto
# Por ejemplo: "Dashboard" vs "dashboard" vs "DASHBOARD"
print("Valores unicos en 'producto':")
print(df["producto"].unique())
print(f"Total: {df['producto'].nunique()} valores distintos")

print("\nValores unicos en 'region':")
print(df["region"].unique())
print(f"Total: {df['region'].nunique()} valores distintos")

In [ ]:
# ============================================================
# ESTADISTICAS DESCRIPTIVAS — buscar valores sospechosos
# ============================================================

# En min y max podemos detectar valores imposibles
# Un precio negativo o unidades en miles son senales de error
df.describe()

**Problemas detectados en el diagnostico:**

1. Nulos en: producto, region, vendedor, unidades, precio_unitario, calificacion, ingreso_total
2. Duplicados exactos (5 filas)
3. Inconsistencias de texto: "Dashboard" vs "dashboard", "Bogota" vs "Bogota", "Medellin" vs "Medellin"
4. Valores atipicos: precio negativo (-200), precios en miles (9500, 12000), unidades = 500

---
## 2. Valores nulos — deteccion y tratamiento

No existe una unica forma de tratar nulos. La decision depende del contexto: que representa la columna, que porcentaje de nulos tiene y que impacto tiene en el analisis.

### 2.1 Estrategia 1: Eliminar filas con nulos

In [ ]:
# ============================================================
# ELIMINAR FILAS CON NULOS — .dropna()
# ============================================================

# Opcion mas agresiva: eliminar CUALQUIER fila que tenga al menos un nulo
# Esto puede reducir mucho el dataset
df_sin_nulos = df.dropna()
print(f"Filas originales: {len(df)}")
print(f"Filas despues de dropna(): {len(df_sin_nulos)}")
print(f"Filas perdidas: {len(df) - len(df_sin_nulos)} ({(len(df) - len(df_sin_nulos))/len(df)*100:.1f}%)")

In [ ]:
# ============================================================
# ELIMINAR FILAS CON NULOS SOLO EN COLUMNAS CRITICAS
# ============================================================

# subset= define en que columnas buscar nulos
# Solo eliminamos si falta el producto o las unidades (columnas criticas)
# Dejamos pasar nulos en calificacion (columna no critica)
df_parcial = df.dropna(subset=["producto", "unidades"])
print(f"Filas despues de dropna(subset): {len(df_parcial)}")
print(f"Filas perdidas: {len(df) - len(df_parcial)}")

### 2.2 Estrategia 2: Imputar (rellenar) nulos

In [ ]:
# ============================================================
# IMPUTACION CON VALOR FIJO
# ============================================================

# Trabajamos sobre una copia para no modificar el original
df_clean = df.copy()

# Para columnas categoricas: rellenar con un valor explicito
# "Desconocido" deja claro que el dato original faltaba
df_clean["vendedor"] = df_clean["vendedor"].fillna("Desconocido")

print("Vendedor despues de fillna:")
print(df_clean["vendedor"].value_counts())

In [ ]:
# ============================================================
# IMPUTACION CON MEDIA, MEDIANA O MODA
# ============================================================

# Para columnas numericas continuas:
# - Media: si la distribucion es simetrica (sin atipicos fuertes)
# - Mediana: si hay atipicos que distorsionan la media

# Veamos la diferencia
media_precio = df_clean["precio_unitario"].mean()
mediana_precio = df_clean["precio_unitario"].median()
print(f"Media de precio_unitario: {media_precio:.2f}")
print(f"Mediana de precio_unitario: {mediana_precio:.2f}")
print(f"Diferencia: {abs(media_precio - mediana_precio):.2f}")
# Si hay mucha diferencia entre media y mediana, hay atipicos
# En ese caso, la mediana es mejor opcion

# Imputar precio con la mediana (mas robusta frente a atipicos)
df_clean["precio_unitario"] = df_clean["precio_unitario"].fillna(mediana_precio)

# Imputar unidades con la mediana
mediana_unidades = df_clean["unidades"].median()
df_clean["unidades"] = df_clean["unidades"].fillna(mediana_unidades)

print(f"\nNulos restantes en precio_unitario: {df_clean['precio_unitario'].isnull().sum()}")
print(f"Nulos restantes en unidades: {df_clean['unidades'].isnull().sum()}")

In [ ]:
# ============================================================
# IMPUTACION CON LA MODA — para variables categoricas
# ============================================================

# .mode() retorna los valores mas frecuentes
# [0] toma el primero en caso de empate
moda_calificacion = df_clean["calificacion"].mode()[0]
print(f"Moda de calificacion: {moda_calificacion}")

df_clean["calificacion"] = df_clean["calificacion"].fillna(moda_calificacion)
print(f"Nulos restantes en calificacion: {df_clean['calificacion'].isnull().sum()}")

In [ ]:
# ============================================================
# IMPUTACION POR INTERPOLACION — para series temporales
# ============================================================

# Ejemplo con datos temporales donde la tendencia importa
serie_ejemplo = pd.Series([100, 110, np.nan, np.nan, 140, 150, np.nan, 170])
print("Serie original:")
print(serie_ejemplo.values)

# .interpolate() llena los nulos con valores intermedios
# method="linear" asume una progresion lineal entre puntos conocidos
serie_interpolada = serie_ejemplo.interpolate(method="linear")
print("\nSerie interpolada:")
print(serie_interpolada.values)

# Nota: la interpolacion es ideal para datos temporales (ventas diarias, temperaturas)
# No tiene sentido para datos categoricos o sin orden

In [ ]:
# ============================================================
# IMPUTACION POR GRUPO — mas precisa que la media global
# ============================================================

# En lugar de imputar con la mediana global, imputamos con
# la mediana del grupo al que pertenece la fila
# Ejemplo: rellenar el precio con la mediana del mismo producto

# Primero, veamos la mediana por producto
print("Mediana de precio por producto:")
print(df.groupby("producto")["precio_unitario"].median())

# .transform() aplica la funcion al grupo y retorna un resultado
# del mismo tamano que la columna original
# Esto permite usarlo directamente con fillna
mediana_por_producto = df.groupby("producto")["precio_unitario"].transform("median")

# Rellenar nulos con la mediana de su producto
df_imputado_grupo = df.copy()
df_imputado_grupo["precio_unitario"] = df_imputado_grupo["precio_unitario"].fillna(mediana_por_producto)

# Si despues de imputar por grupo quedan nulos (porque el producto tambien era nulo),
# rellenar con la mediana global como respaldo
df_imputado_grupo["precio_unitario"] = df_imputado_grupo["precio_unitario"].fillna(
    df["precio_unitario"].median()
)

print(f"\nNulos restantes: {df_imputado_grupo['precio_unitario'].isnull().sum()}")

### Resumen de estrategias para nulos

| Estrategia | Cuando usarla | Riesgo |
|---|---|---|
| **Eliminar filas** | Pocos nulos (<5%), datos no criticos | Perder informacion valiosa |
| **Valor fijo** | Categoricas, cuando "Desconocido" tiene sentido | Introduce un valor artificial |
| **Media** | Numericas con distribucion simetrica | Se distorsiona con atipicos |
| **Mediana** | Numericas con atipicos o distribucion asimetrica | Menos sensible a la forma real |
| **Moda** | Categoricas o discretas con valor dominante claro | Sesga hacia el valor mas comun |
| **Interpolacion** | Series temporales con tendencia | Asume continuidad que puede no existir |
| **Por grupo** | Cuando el valor depende de otra variable | Requiere que el grupo no tenga nulos |

---
## 3. Duplicados — identificacion y eliminacion

Los duplicados pueden venir de cargas dobles, joins mal hechos o errores de integracion. Hay dos tipos: exactos (toda la fila es identica) y parciales (misma entidad con datos ligeramente distintos).

In [ ]:
# ============================================================
# DETECTAR DUPLICADOS EXACTOS
# ============================================================

# .duplicated() marca True las filas que son copia exacta de una anterior
print(f"Duplicados exactos: {df_clean.duplicated().sum()}")

# Ver las filas duplicadas junto con sus originales
# keep=False marca TODAS las ocurrencias (tanto la original como las copias)
mask_duplicados = df_clean.duplicated(keep=False)
df_clean[mask_duplicados].sort_values("id_venta").head(10)

In [ ]:
# ============================================================
# ELIMINAR DUPLICADOS EXACTOS
# ============================================================

print(f"Filas antes: {len(df_clean)}")

# .drop_duplicates() elimina filas identicas
# keep="first" conserva la primera ocurrencia (por defecto)
# keep="last" conservaria la ultima
# keep=False eliminaria TODAS las ocurrencias
df_clean = df_clean.drop_duplicates(keep="first")

print(f"Filas despues: {len(df_clean)}")
print(f"Filas eliminadas: {len(df) - len(df_clean) + (len(df) - 200)}")

In [ ]:
# ============================================================
# DUPLICADOS PARCIALES — por subconjunto de columnas
# ============================================================

# A veces dos filas representan la misma venta pero con datos ligeramente distintos
# Ejemplo: misma fecha, producto y region pero diferente vendedor (error de captura)

# subset= define que columnas considerar para detectar duplicados
duplicados_parciales = df_clean.duplicated(
    subset=["fecha", "producto", "region"],
    keep=False
)

n_parciales = duplicados_parciales.sum()
print(f"Duplicados parciales (misma fecha+producto+region): {n_parciales}")

if n_parciales > 0:
    print("\nEjemplos:")
    print(df_clean[duplicados_parciales].sort_values(
        ["fecha", "producto", "region"]
    ).head(10))

---
## 4. Valores atipicos — deteccion y decision

Un valor atipico (outlier) no siempre es un error. Puede ser un dato real pero extremo, o puede ser un error de captura. La decision de que hacer con el depende del contexto y del impacto en el analisis.

### 4.1 Deteccion con IQR (Rango Intercuartilico)

In [ ]:
# ============================================================
# METODO IQR — el mas usado para deteccion de atipicos
# ============================================================

# IQR = Q3 - Q1
# Un valor es atipico si esta por debajo de Q1 - 1.5*IQR
# o por encima de Q3 + 1.5*IQR
# Este es el mismo criterio que usa un boxplot para los "bigotes"

def detectar_atipicos_iqr(serie, factor=1.5):
    """
    Detecta valores atipicos usando el metodo IQR.
    
    Parametros:
        serie (pd.Series): columna numerica
        factor (float): multiplicador del IQR (1.5 es estandar, 3.0 para extremos)
    
    Retorna:
        pd.Series: mascara booleana (True = atipico)
    """
    Q1 = serie.quantile(0.25)   # Percentil 25
    Q3 = serie.quantile(0.75)   # Percentil 75
    IQR = Q3 - Q1               # Rango intercuartilico
    
    limite_inferior = Q1 - factor * IQR
    limite_superior = Q3 + factor * IQR
    
    print(f"  Q1: {Q1:.2f}")
    print(f"  Q3: {Q3:.2f}")
    print(f"  IQR: {IQR:.2f}")
    print(f"  Limites: [{limite_inferior:.2f}, {limite_superior:.2f}]")
    
    # Retornar True donde el valor esta fuera de los limites
    return (serie < limite_inferior) | (serie > limite_superior)

# Aplicar a precio_unitario
print("Atipicos en precio_unitario:")
mask_atipicos_precio = detectar_atipicos_iqr(df_clean["precio_unitario"])
print(f"  Encontrados: {mask_atipicos_precio.sum()}")

# Ver los valores atipicos
print("\nValores atipicos:")
print(df_clean.loc[mask_atipicos_precio, ["id_venta", "producto", "precio_unitario"]])

### 4.2 Deteccion con Z-Score

In [ ]:
# ============================================================
# METODO Z-SCORE — distancia en desviaciones estandar
# ============================================================

# Z = (x - media) / desviacion_estandar
# Un valor con |Z| > 3 esta a mas de 3 desviaciones de la media
# Esto es atipico en una distribucion normal

def detectar_atipicos_zscore(serie, umbral=3):
    """
    Detecta atipicos usando Z-Score.
    
    Parametros:
        serie (pd.Series): columna numerica
        umbral (float): numero de desviaciones estandar (3 es estandar)
    
    Retorna:
        pd.Series: mascara booleana (True = atipico)
    """
    media = serie.mean()
    std = serie.std()
    
    # Calcular z-score para cada valor
    z_scores = (serie - media) / std
    
    print(f"  Media: {media:.2f}")
    print(f"  Desviacion estandar: {std:.2f}")
    print(f"  Umbral: |Z| > {umbral}")
    
    return z_scores.abs() > umbral

# Aplicar a precio_unitario
print("Atipicos en precio_unitario (Z-Score):")
mask_zscore = detectar_atipicos_zscore(df_clean["precio_unitario"])
print(f"  Encontrados: {mask_zscore.sum()}")

print("\nAtipicos en unidades (Z-Score):")
mask_zscore_u = detectar_atipicos_zscore(df_clean["unidades"])
print(f"  Encontrados: {mask_zscore_u.sum()}")

### 4.3 Que hacer con los atipicos

In [ ]:
# ============================================================
# OPCION A: ELIMINAR las filas con atipicos
# ============================================================

# Usar cuando estamos seguros de que son errores
# El operador ~ invierte la mascara (True -> False y viceversa)
df_sin_atipicos = df_clean[~mask_atipicos_precio].copy()
print(f"Filas eliminadas: {len(df_clean) - len(df_sin_atipicos)}")
print(f"Filas restantes: {len(df_sin_atipicos)}")

In [ ]:
# ============================================================
# OPCION B: RECORTAR (clipping) a los limites
# ============================================================

# Reemplazar atipicos con el valor limite mas cercano
# Esto conserva la fila pero limita el valor extremo

Q1 = df_clean["precio_unitario"].quantile(0.25)
Q3 = df_clean["precio_unitario"].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

# .clip() limita los valores al rango [lower, upper]
# Los valores por debajo de lower se convierten en lower
# Los valores por encima de upper se convierten en upper
df_clean["precio_clipped"] = df_clean["precio_unitario"].clip(lower=lim_inf, upper=lim_sup)

# Comparar original vs clipped en los atipicos
print("Comparacion en valores atipicos:")
print(df_clean.loc[mask_atipicos_precio, ["precio_unitario", "precio_clipped"]])

In [ ]:
# ============================================================
# OPCION C: REEMPLAZAR con la mediana
# ============================================================

# Similar a la imputacion de nulos, pero para valores atipicos
mediana = df_clean["precio_unitario"].median()

df_clean["precio_corregido"] = df_clean["precio_unitario"].copy()
# np.where(condicion, valor_si_true, valor_si_false)
df_clean["precio_corregido"] = np.where(
    mask_atipicos_precio,   # Donde hay atipicos
    mediana,                # Reemplazar con la mediana
    df_clean["precio_unitario"]  # Dejar el original si no es atipico
)

print("Comparacion en valores atipicos:")
print(df_clean.loc[mask_atipicos_precio, ["precio_unitario", "precio_clipped", "precio_corregido"]])

In [ ]:
# ============================================================
# TRATAR VALORES IMPOSIBLES — reglas de negocio
# ============================================================

# Algunos atipicos no necesitan metodos estadisticos,
# son simplemente valores imposibles por logica de negocio

# Precio negativo: imposible -> convertir a nulo
df_clean.loc[df_clean["precio_unitario"] < 0, "precio_unitario"] = np.nan

# Unidades mayores a 200: imposible en este negocio -> convertir a nulo
df_clean.loc[df_clean["unidades"] > 200, "unidades"] = np.nan

# Luego se imputan con la estrategia que prefiramos
df_clean["precio_unitario"] = df_clean["precio_unitario"].fillna(
    df_clean["precio_unitario"].median()
)
df_clean["unidades"] = df_clean["unidades"].fillna(
    df_clean["unidades"].median()
)

print("Rango de precio_unitario despues de limpiar:")
print(f"  Min: {df_clean['precio_unitario'].min():.2f}")
print(f"  Max: {df_clean['precio_unitario'].max():.2f}")
print(f"\nRango de unidades despues de limpiar:")
print(f"  Min: {df_clean['unidades'].min():.0f}")
print(f"  Max: {df_clean['unidades'].max():.0f}")

---
## 5. Normalizacion y encoding de variables

Despues de limpiar, necesitamos estandarizar los datos: unificar texto inconsistente, escalar variables numericas y convertir categorias a numeros para que los algoritmos puedan procesarlos.

### 5.1 Estandarizacion de texto

In [ ]:
# ============================================================
# ESTANDARIZAR COLUMNAS DE TEXTO
# ============================================================

# Problema: "Dashboard", "dashboard" y "DASHBOARD" son el mismo producto
# pero pandas los trata como valores distintos

print("ANTES de estandarizar:")
print(df_clean["producto"].value_counts())

# Paso 1: convertir a minusculas
# .str.lower() convierte todos los caracteres a minusculas
df_clean["producto"] = df_clean["producto"].str.lower()

# Paso 2: eliminar espacios al inicio y al final
# .str.strip() quita espacios, tabs y saltos de linea sobrantes
df_clean["producto"] = df_clean["producto"].str.strip()

print("\nDESPUES de estandarizar:")
print(df_clean["producto"].value_counts())

In [ ]:
# ============================================================
# CORREGIR TILDES Y VARIANTES — con .replace()
# ============================================================

print("ANTES — region:")
print(df_clean["region"].value_counts())

# Mapeo de correcciones: {valor_incorrecto: valor_correcto}
correcciones_region = {
    "Bogota": "Bogota",     # Sin tilde -> con tilde (o viceversa, elegir uno)
    "Bogotá": "Bogota",     # Unificar a una sola forma
    "Medellin": "Medellin",
    "Medellín": "Medellin",
}

# .replace() busca cada clave y la sustituye por su valor
df_clean["region"] = df_clean["region"].replace(correcciones_region)

print("\nDESPUES — region:")
print(df_clean["region"].value_counts())

### 5.2 Escalado de variables numericas

In [ ]:
# ============================================================
# MIN-MAX SCALING — escalar al rango [0, 1]
# ============================================================

# Formula: X_norm = (X - X_min) / (X_max - X_min)
# El valor minimo se convierte en 0, el maximo en 1
# Todos los demas quedan proporcionalmente entre 0 y 1

def min_max_scaling(serie):
    """Escala una serie al rango [0, 1]"""
    return (serie - serie.min()) / (serie.max() - serie.min())

# Aplicar a precio_unitario y unidades
df_clean["precio_norm"] = min_max_scaling(df_clean["precio_unitario"])
df_clean["unidades_norm"] = min_max_scaling(df_clean["unidades"])

print("Precio unitario — original vs normalizado:")
print(df_clean[["precio_unitario", "precio_norm"]].describe().round(3))

In [ ]:
# ============================================================
# Z-SCORE STANDARDIZATION — media=0, desviacion=1
# ============================================================

# Formula: Z = (X - media) / desviacion_estandar
# El resultado tiene media 0 y desviacion estandar 1
# Los valores se interpretan como "distancia en desviaciones de la media"

def z_score_scaling(serie):
    """Estandariza una serie a media=0, std=1"""
    return (serie - serie.mean()) / serie.std()

df_clean["precio_zscore"] = z_score_scaling(df_clean["precio_unitario"])
df_clean["unidades_zscore"] = z_score_scaling(df_clean["unidades"])

print("Precio unitario — Z-Score:")
print(df_clean[["precio_unitario", "precio_zscore"]].describe().round(3))

**Cuando usar cada uno:**

| Metodo | Rango resultante | Cuando usarlo |
|---|---|---|
| **Min-Max** | [0, 1] | Cuando necesitas valores acotados. Redes neuronales, distancias. |
| **Z-Score** | Sin limite fijo (tipicamente -3 a 3) | Cuando importa la distribucion. Regresion, PCA, clustering. |

### 5.3 Encoding de variables categoricas

In [ ]:
# ============================================================
# LABEL ENCODING — convertir categorias a numeros enteros
# ============================================================

# Cada categoria recibe un numero unico
# Util para variables ordinales (donde el orden importa)
# Ejemplo: satisfaccion -> bajo=1, medio=2, alto=3

# Con pandas: .map() con un diccionario
mapeo_calificacion = {
    1.0: "Muy mala",
    2.0: "Mala",
    3.0: "Regular",
    4.0: "Buena",
    5.0: "Excelente"
}

df_clean["calificacion_texto"] = df_clean["calificacion"].map(mapeo_calificacion)
print("Calificacion como texto:")
print(df_clean["calificacion_texto"].value_counts().sort_index())

# El proceso inverso: de texto a numero
# Usando .astype("category").cat.codes
df_clean["producto_code"] = df_clean["producto"].astype("category").cat.codes
print("\nProducto como codigo:")
# Mostrar el mapeo
categorias = df_clean["producto"].astype("category").cat.categories
for i, cat in enumerate(categorias):
    print(f"  {i} = {cat}")

In [ ]:
# ============================================================
# ONE-HOT ENCODING — una columna binaria por categoria
# ============================================================

# Crea una columna nueva por cada valor unico de la variable
# Cada columna tiene 1 si la fila pertenece a esa categoria, 0 si no
# Necesario para variables nominales (sin orden) en modelos de ML

# pd.get_dummies() hace todo el trabajo
df_onehot = pd.get_dummies(
    df_clean,
    columns=["producto", "region"],  # Columnas a codificar
    prefix=["prod", "reg"],          # Prefijo para las nuevas columnas
    drop_first=False                 # True para evitar multicolinealidad en regresion
)

# Mostrar las columnas nuevas
cols_nuevas = [c for c in df_onehot.columns if c.startswith("prod_") or c.startswith("reg_")]
print("Columnas creadas por One-Hot Encoding:")
for col in cols_nuevas:
    print(f"  {col}")

print(f"\nDimensiones originales: {df_clean.shape}")
print(f"Dimensiones con one-hot: {df_onehot.shape}")

# Ver un ejemplo
df_onehot[cols_nuevas].head()

In [ ]:
# ============================================================
# ORDINAL ENCODING — para categorias con orden natural
# ============================================================

# Cuando la variable tiene un orden logico:
# bajo < medio < alto
# Le asignamos numeros que respeten ese orden

mapeo_ordinal = {
    "Muy mala": 1,
    "Mala": 2,
    "Regular": 3,
    "Buena": 4,
    "Excelente": 5
}

df_clean["calificacion_ordinal"] = df_clean["calificacion_texto"].map(mapeo_ordinal)

print("Encoding ordinal:")
print(df_clean[["calificacion_texto", "calificacion_ordinal"]].drop_duplicates().sort_values(
    "calificacion_ordinal"
))

### Resumen de encoding

| Metodo | Que hace | Cuando usarlo |
|---|---|---|
| **Label Encoding** | Categoria -> numero entero | Variables ordinales o arboles de decision |
| **One-Hot Encoding** | Categoria -> N columnas binarias | Variables nominales en modelos lineales |
| **Ordinal Encoding** | Categoria -> numero con orden | Variables con jerarquia natural |

---
## 6. Pipeline completo de limpieza

En la practica, todos estos pasos se encapsulan en una funcion que recibe datos crudos y retorna datos limpios. Esto permite reutilizar el proceso cada vez que lleguen datos nuevos.

In [ ]:
# ============================================================
# FUNCION DE LIMPIEZA COMPLETA
# ============================================================

def limpiar_datos_ventas(df_raw):
    """
    Pipeline de limpieza para el dataset de ventas.
    Recibe datos crudos y retorna datos listos para analisis.
    
    Pasos:
        1. Eliminar duplicados exactos
        2. Estandarizar texto
        3. Corregir valores imposibles
        4. Imputar nulos
        5. Recalcular columnas derivadas
    """
    # Trabajar sobre copia
    df = df_raw.copy()
    filas_inicio = len(df)
    
    # --- Paso 1: Duplicados ---
    df = df.drop_duplicates(keep="first")
    print(f"[1] Duplicados eliminados: {filas_inicio - len(df)}")
    
    # --- Paso 2: Estandarizar texto ---
    # Producto: minusculas + strip
    df["producto"] = df["producto"].str.lower().str.strip()
    
    # Region: unificar variantes
    correcciones = {
        "Bogotá": "Bogota",
        "Medellín": "Medellin",
    }
    df["region"] = df["region"].replace(correcciones)
    print(f"[2] Texto estandarizado")
    
    # --- Paso 3: Valores imposibles ---
    # Precio negativo -> nulo
    n_negativos = (df["precio_unitario"] < 0).sum()
    df.loc[df["precio_unitario"] < 0, "precio_unitario"] = np.nan
    
    # Unidades > 200 -> nulo
    n_exceso = (df["unidades"] > 200).sum()
    df.loc[df["unidades"] > 200, "unidades"] = np.nan
    print(f"[3] Valores imposibles corregidos: {n_negativos} precios negativos, {n_exceso} unidades excesivas")
    
    # --- Paso 4: Imputar nulos ---
    # Categoricas: "desconocido"
    for col in ["producto", "region", "vendedor"]:
        n_nulos = df[col].isnull().sum()
        if n_nulos > 0:
            df[col] = df[col].fillna("desconocido")
    
    # Numericas: mediana
    for col in ["unidades", "precio_unitario", "calificacion"]:
        n_nulos = df[col].isnull().sum()
        if n_nulos > 0:
            mediana = df[col].median()
            df[col] = df[col].fillna(mediana)
    
    nulos_restantes = df.isnull().sum().sum()
    print(f"[4] Nulos imputados. Restantes: {nulos_restantes}")
    
    # --- Paso 5: Recalcular columnas derivadas ---
    df["ingreso_total"] = df["unidades"] * df["precio_unitario"]
    print(f"[5] Columnas recalculadas")
    
    # --- Resetear indice ---
    df = df.reset_index(drop=True)
    
    print(f"\nResultado: {len(df)} filas, {df.shape[1]} columnas")
    return df

In [ ]:
# ============================================================
# EJECUTAR EL PIPELINE
# ============================================================

# Volvemos a cargar los datos crudos originales para demostrar
# que el pipeline funciona de principio a fin

# Re-crear el dataset crudo (reutilizando el codigo de la seccion 0)
np.random.seed(42)
n = 200
datos_crudos = {
    "id_venta": list(range(1, n + 1)),
    "fecha": pd.date_range("2025-01-01", periods=n, freq="D"),
    "producto": np.random.choice(
        ["Dashboard", "Reporte", "API", "App Web", "dashboard", "REPORTE", None], size=n
    ),
    "region": np.random.choice(
        ["Eje Cafetero", "Bogota", "Bogotá", "Medellín", "Medellin", "Cali", None], size=n
    ),
    "vendedor": np.random.choice(
        ["Ana García", "Carlos López", "María Ruiz", "Pedro Martín", None], size=n
    ),
    "unidades": np.random.randint(1, 50, size=n).astype(float),
    "precio_unitario": np.round(np.random.uniform(50, 500, size=n), 2),
    "calificacion": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n),
}
df_crudo = pd.DataFrame(datos_crudos)
indices_nulos = np.random.choice(n, size=15, replace=False)
df_crudo.loc[indices_nulos, "unidades"] = np.nan
df_crudo.loc[5, "precio_unitario"] = 9500.0
df_crudo.loc[42, "precio_unitario"] = -200.0
df_crudo.loc[150, "unidades"] = 500.0
filas_dup = df_crudo.iloc[[10, 10, 25]].copy()
df_crudo = pd.concat([df_crudo, filas_dup], ignore_index=True)
df_crudo["ingreso_total"] = df_crudo["unidades"] * df_crudo["precio_unitario"]

print("=" * 50)
print("  EJECUTANDO PIPELINE DE LIMPIEZA")
print("=" * 50)
df_final = limpiar_datos_ventas(df_crudo)

In [ ]:
# ============================================================
# VERIFICACION FINAL
# ============================================================

print("=== Verificacion post-limpieza ===")
print(f"\nNulos totales: {df_final.isnull().sum().sum()}")
print(f"Duplicados: {df_final.duplicated().sum()}")
print(f"\nValores unicos en producto: {df_final['producto'].nunique()}")
print(df_final["producto"].value_counts())
print(f"\nValores unicos en region: {df_final['region'].nunique()}")
print(df_final["region"].value_counts())
print(f"\nRango de precio_unitario: [{df_final['precio_unitario'].min():.2f}, {df_final['precio_unitario'].max():.2f}]")
print(f"Rango de unidades: [{df_final['unidades'].min():.0f}, {df_final['unidades'].max():.0f}]")

In [ ]:
# ============================================================
# GUARDAR RESULTADO
# ============================================================

df_final.to_csv("ventas_limpias.csv", index=False)
print("Archivo guardado: ventas_limpias.csv")
print(f"Dimensiones finales: {df_final.shape}")

---
## Resumen de la subseccion

| Tema | Lo esencial |
|---|---|
| **Diagnostico** | .info(), .isnull().sum(), .duplicated(), .describe(), .value_counts() para entender el dataset |
| **Nulos** | dropna() para eliminar, fillna() para imputar (media, mediana, moda, por grupo, interpolacion) |
| **Duplicados** | .duplicated() para detectar, .drop_duplicates() para eliminar (exactos o por subset) |
| **Atipicos** | IQR y Z-Score para detectar. Eliminar, recortar con .clip() o reemplazar segun el contexto |
| **Normalizacion** | Min-Max al rango [0,1], Z-Score a media=0 std=1 |
| **Encoding** | Label encoding para ordinales, One-Hot para nominales, mapeo manual para ordenes especificos |
| **Pipeline** | Encapsular todo en una funcion reutilizable que recibe datos crudos y retorna datos limpios |

### Siguiente paso
En el **Notebook 1.3** tomaremos estos datos limpios y los visualizaremos con Matplotlib, Seaborn y Plotly para encontrar patrones y comunicar hallazgos.